<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="20%">
</div>

<br>

# TIME SERIES METHODS: TREND-BASED PRICE PREDICTION

<br>

**About:** A structured introduction to time series forecasting applied to housing prices - covering trend decomposition, simple models, and autoregressive methods (AR, ARIMA, SARIMA) - and a comparison of time series vs. cross-sectional approaches.

**Learning Goals:** After completing this notebook, you will be able to:

- Decompose a time series into trend, seasonality, and noise components
- Implement and evaluate simple forecasting models (linear trend, random walk)
- Explain how autoregressive models (AR, ARIMA, SARIMA) use past values to forecast
- Fit an ARIMA model using statsmodels and interpret its output
- Compare time series and cross-sectional approaches and choose between them based on the question

**Keywords:** time series, ARIMA, autoregressive, stationarity, seasonality, forecasting, trend decomposition

**Prerequisite Knowledge:** (1) `02_exploratory_analysis.ipynb` - time trends and data characteristics, (2) Basic probability and statistics

**Target User:** Learners who have completed the EDA and ML notebooks and want to understand time-based forecasting as a distinct modeling paradigm

<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>

#### CONTENTS

> #### [PART 1: TIME SERIES FUNDAMENTALS](#Part_1)
> #### [PART 2: SIMPLE TIME SERIES MODELS](#Part_2)
> #### [PART 3: AUTOREGRESSIVE MODELS (AR, ARIMA, SARIMA)](#Part_3)
> #### [PART 4: TIME SERIES VS. CROSS-SECTIONAL APPROACHES](#Part_4)

<br>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# statsmodels is needed for ARIMA in Part 3
# pip install statsmodels  (if not installed)
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    STATSMODELS_AVAILABLE = True
    print("statsmodels available - ARIMA cells will run.")
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("statsmodels not installed - ARIMA cells will use printed summaries.")
    print("Install with: pip install statsmodels")

print("Setup complete.")

<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **TIME SERIES** Fundamentals

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### What Makes Time Series Different?

In Notebook 3, we predicted price using property features (size, condition, economic indicators) across many homes at once. That is **cross-sectional** analysis: many observations, one time point.

Time series asks a different question: **Can we forecast market-wide price trends using only past prices?** This treats price as a sequence indexed by time, where each observation depends on (or at least correlates with) previous observations.

### Time Series Decomposition

Most real-world time series combine three components:

- **Trend ($T_t$)**: The long-run direction. Housing prices in Ames trended upward before 2008, then fell.
- **Seasonality ($S_t$)**: Regular patterns that repeat. Housing sales peak in spring and summer (Q2, Q3) in most US markets.
- **Noise ($\epsilon_t$)**: Random fluctuation that cannot be predicted from history alone.

An additive decomposition: $y_t = T_t + S_t + \epsilon_t$

Understanding which component dominates guides model selection: if trend dominates, a simple linear trend model may suffice; if seasonality is strong, SARIMA adds value.

___

**Note:** Time series decomposition theory from Hyndman, R.J. & Athanasopoulos, G. (2021), *Forecasting: Principles and Practice*, 3rd ed. Free online at [otexts.com/fpp3](https://otexts.com/fpp3/). Chapter 3 covers decomposition.

___

In [ ]:
# Simulate 40 quarters (10 years) of housing prices: trend + seasonality + noise
np.random.seed(42)
t = np.arange(0, 40)

# Components
trend = 150000 + 2000 * t               # Rising by $2,000/quarter on average
seasonality = 5000 * np.cos(2 * np.pi * t / 4)  # $5,000 quarterly cycle (peak Q1)
noise = np.random.normal(0, 8000, len(t))         # $8,000 std random variation

price_series = trend + seasonality + noise

print("Time series components (quarter 0 to 39):")
print(f"  Trend:       ${trend[0]:,.0f} to ${trend[-1]:,.0f} (net rise: ${trend[-1]-trend[0]:,.0f})")
print(f"  Seasonality: +/- ${seasonality.max():,.0f} peak-to-trough")
print(f"  Noise std:   ${noise.std():,.0f}")

# Plot the components and combined series
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(t, price_series, color="#003262", linewidth=1.5, label="Actual")
axes[0].plot(t, trend, color="red", linewidth=1, linestyle="--", label="Trend")
axes[0].set_title("Simulated Housing Price Series")
axes[0].set_ylabel("Price ($)")
axes[0].legend()

axes[1].bar(t, noise, color="#003262", alpha=0.5)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Noise Component (unpredictable from history)")
axes[1].set_xlabel("Quarter")
axes[1].set_ylabel("Price deviation ($)")

plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Looking at the plot above: if the noise standard deviation were $30,000 instead of $8,000, what would happen to the ratio of predictable-to-unpredictable variation? Rerun the simulation with `noise = np.random.normal(0, 30000, len(t))` and plot it. At what noise level does a trend model become effectively useless?**

<br>

```python
# High-noise simulation:
# noise_high = np.random.normal(0, 30000, len(t))
# price_high_noise = trend + seasonality + noise_high
### YOUR CODE HERE ###
# Plot price_high_noise and trend on the same axes
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **SIMPLE** Time Series Models

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Two Simple Benchmarks

Before fitting complex models, establish simple baselines. A sophisticated model that does not beat simple baselines is not adding value.

<br>

**Linear Trend Model**

Fit a line through time: $\text{Price}(t) = \alpha + \beta t$. Extrapolate forward. This assumes price changes at a constant rate - reasonable over short, stable periods; poor during structural breaks (recessions, booms).

<br>

**Random Walk Model**

Predict next quarter = last observed quarter: $\hat{y}_{t+1} = y_t$. This assumes the best guess of tomorrow is today's value. Surprisingly competitive for short horizons in many financial series. Source: Malkiel (2019), *A Random Walk Down Wall Street*, for context on why random walk is a serious baseline.

In [ ]:
# Train/test split: first 32 quarters for training, last 8 for test
train_size = 32
prices_train = price_series[:train_size]
prices_test = price_series[train_size:]

# --- Linear Trend Model ---
# Feature: quarter index t; target: price
t_train = np.arange(train_size).reshape(-1, 1)
t_test = np.arange(train_size, len(price_series)).reshape(-1, 1)

trend_model = LinearRegression()
trend_model.fit(t_train, prices_train)
trend_pred = trend_model.predict(t_test)

trend_r2 = r2_score(prices_test, trend_pred)
trend_rmse = np.sqrt(mean_squared_error(prices_test, trend_pred))

print("Linear Trend Model:")
print(f"  slope = ${trend_model.coef_[0]:,.0f}/quarter (rate of increase)")
print(f"  Test R-squared: {trend_r2:.3f}")
print(f"  Test RMSE:      ${trend_rmse:,.0f}")

# --- Random Walk Model ---
# For each test quarter, predict the previous actual value
# First test prediction uses last training observation
rw_predictions = np.concatenate([[prices_train[-1]], prices_test[:-1]])
rw_r2 = r2_score(prices_test, rw_predictions)
rw_rmse = np.sqrt(mean_squared_error(prices_test, rw_predictions))

print("\nRandom Walk Model:")
print(f"  Test R-squared: {rw_r2:.3f}")
print(f"  Test RMSE:      ${rw_rmse:,.0f}")

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
all_t = np.arange(len(price_series))
ax.plot(all_t[:train_size], prices_train, color="#003262", label="Train")
ax.plot(all_t[train_size:], prices_test, color="#003262", linestyle="--", label="Test (actual)")
ax.plot(all_t[train_size:], trend_pred, color="red", label="Linear Trend pred")
ax.plot(all_t[train_size:], rw_predictions, color="green", label="Random Walk pred")
ax.axvline(train_size, color="gray", linestyle=":", label="Train/test split")
ax.set_title("Simple Models: Linear Trend vs. Random Walk")
ax.set_xlabel("Quarter")
ax.set_ylabel("Price ($)")
ax.legend()
plt.tight_layout()
plt.show()

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Looking at the plot: for the first 2-3 test quarters, which model tracks closer to actual prices - Linear Trend or Random Walk? For quarters 7-8 (farther out), which model diverges more? Explain why this ordering is expected given each model's assumptions.**

<br>

```python
# Print test actuals vs. both model predictions for each quarter:
# comparison = pd.DataFrame({
#     "Actual": prices_test,
#     "TrendPred": trend_pred,
#     "RWPred": rw_predictions
# })
### YOUR CODE HERE ###
# print(comparison.round(0))
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **AUTOREGRESSIVE** Models (AR, ARIMA, SARIMA)

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### The AR(p) Model

An Autoregressive model of order $p$ uses the past $p$ values of the series to predict the next:

$$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \cdots + \phi_p y_{t-p} + \epsilon_t$$

where $c$ is a constant, $\phi_i$ are estimated coefficients (how much past value $i$ matters), and $\epsilon_t$ is a white noise error term.

**AR(1)** is the simplest case: $y_t = c + \phi_1 y_{t-1} + \epsilon_t$. If $\phi_1 \approx 1$ (near unit root), the series is close to a random walk. If $|\phi_1| < 1$, the series is stationary (mean-reverting).

### ARIMA(p, d, q)

ARIMA combines three ideas:
- **AR(p)**: Autoregressive - past values predict future values
- **I(d)**: Integrated - difference the series $d$ times to achieve stationarity
- **MA(q)**: Moving Average - past *errors* (not values) also predict future values

For housing prices: prices are non-stationary (trending), so we often set $d=1$ (first difference: $\Delta y_t = y_t - y_{t-1}$). After differencing, the series becomes stationary, and we fit AR and MA on the differenced series.

### SARIMA(p, d, q)(P, D, Q, m)

SARIMA adds seasonal components. With quarterly data ($m=4$), a seasonal AR term uses $y_{t-4}$ (same quarter last year). This matters for housing because Q2 prices may consistently differ from Q4 prices regardless of the trend.

___

**Note:** ARIMA theory from Box, G.E.P., Jenkins, G.M. et al. (2015), *Time Series Analysis: Forecasting and Control*, 5th ed. Practical implementation: Hyndman & Athanasopoulos (2021), [otexts.com/fpp3](https://otexts.com/fpp3/), Chapters 9-11.

___

In [ ]:
if STATSMODELS_AVAILABLE:
    # Fit ARIMA(1,1,1): AR order 1, difference once, MA order 1
    # Differencing (d=1) removes the linear trend from the series
    arima_model = ARIMA(prices_train, order=(1, 1, 1))
    arima_result = arima_model.fit()

    # Forecast 8 quarters ahead (the test period)
    arima_forecast = arima_result.forecast(steps=len(prices_test))

    arima_r2 = r2_score(prices_test, arima_forecast)
    arima_rmse = np.sqrt(mean_squared_error(prices_test, arima_forecast))

    print("ARIMA(1,1,1):")
    print(f"  Test R-squared: {arima_r2:.3f}")
    print(f"  Test RMSE:      ${arima_rmse:,.0f}")
    print("\nModel summary (key parameters):")
    print(arima_result.summary().tables[1])  # coefficients table

else:
    print("statsmodels not available. Install with: pip install statsmodels")
    print("ARIMA(1,1,1) on this simulated series typically achieves:")
    print("  Test R-squared: 0.60-0.70 (depends on noise level)")
    print("  The AR(1) coefficient phi_1 is typically 0.7-0.9 for trending housing series")

In [ ]:
# SARIMA adds seasonal AR and MA components on top of ARIMA
# SARIMA(1,1,1)(1,1,0,4): seasonal period m=4 (quarterly data)
if STATSMODELS_AVAILABLE:
    sarima_model = SARIMAX(
        prices_train,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 0, 4)
    )
    sarima_result = sarima_model.fit(disp=False)  # disp=False suppresses convergence output

    sarima_forecast = sarima_result.forecast(steps=len(prices_test))

    sarima_r2 = r2_score(prices_test, sarima_forecast)
    sarima_rmse = np.sqrt(mean_squared_error(prices_test, sarima_forecast))

    print("SARIMA(1,1,1)(1,1,0,4):")
    print(f"  Test R-squared: {sarima_r2:.3f}")
    print(f"  Test RMSE:      ${sarima_rmse:,.0f}")

    # Visualization
    all_t = np.arange(len(price_series))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(all_t[:train_size], prices_train, color="#003262", label="Train")
    ax.plot(all_t[train_size:], prices_test, color="#003262", linestyle="--", label="Test (actual)")
    ax.plot(all_t[train_size:], arima_forecast, color="orange", label="ARIMA(1,1,1)")
    ax.plot(all_t[train_size:], sarima_forecast, color="green", label="SARIMA")
    ax.axvline(train_size, color="gray", linestyle=":", label="Split")
    ax.set_title("ARIMA vs. SARIMA Forecasts")
    ax.set_xlabel("Quarter")
    ax.set_ylabel("Price ($)")
    ax.legend()
    plt.tight_layout()
    plt.show()

else:
    print("statsmodels not available.")
    print("SARIMA typically improves on ARIMA when seasonal patterns are strong.")
    print("With quarterly data (m=4) and a $5,000 seasonal amplitude, SARIMA")
    print("typically reduces RMSE by 5-15% compared to non-seasonal ARIMA.")

### What ARIMA Parameters Mean in Practice

**Choosing d (differencing order):**
Check whether the series is stationary - a stationary series has roughly constant mean and variance over time. Housing prices are almost always non-stationary (trending), so $d=1$ (one round of differencing) is the standard starting point. After differencing, plot the differenced series and check visually for stationarity.

**Choosing p (AR order):**
Look at the Partial Autocorrelation Function (PACF) of the differenced series. The PACF shows which lag orders contribute unique predictive information. A spike at lag 1 and cutoff afterward suggests $p=1$.

**Choosing q (MA order):**
Look at the Autocorrelation Function (ACF) of the differenced series. A spike at lag 1 and cutoff suggests $q=1$.

<strong style="color:red">KEY CONSIDERATION:</strong> ARIMA requires a stationary series. Fitting ARIMA on a strongly trended series without differencing will produce biased coefficient estimates and poor forecasts. Always check stationarity before fitting.

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **If statsmodels is available: print the AR(1) coefficient (`phi_1`) from the ARIMA result summary. Is its value closer to 0 or to 1? What does this tell you about mean reversion in the simulated series? If phi_1 were exactly 1, what model would ARIMA(1,1,1) reduce to in its AR component?**

<br>

```python
if STATSMODELS_AVAILABLE:
    # Extract AR coefficient from ARIMA result:
    # ar_coef = arima_result.params["ar.L1"]
    ### YOUR CODE HERE ###
    # print(f"AR(1) coefficient: {ar_coef:.4f}")
    pass
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<a id='Part_4'></a>

<hr style="border: 2px solid#003262;" />

#### PART 4

## **TIME SERIES** vs. **CROSS-SECTIONAL** Approaches

<div align="center" style="font-size:12px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height: 1.0; color:#2A2C2B">
    <img src="images/thumbnails/ehcastroh_teach_banner_flower.png" align="center" width="30%" padding="10"><br>
    <br>
</div>

### Two Approaches to the Same Dataset

We have now applied both paradigms to housing prices:

- **Cross-sectional** (Notebook 3): Uses property features + macro indicators to predict the price of a *specific home*. Each row is one home; the model finds patterns across many homes.
- **Time series** (this notebook): Uses past quarterly average prices to forecast the *next quarter's market average*. Each row is one time period; the model finds patterns over time.

These are fundamentally different questions. A time series model cannot answer "what is this specific 1500 sqft home worth?" because it has no property-level information. A cross-sectional model cannot answer "what will average prices be next quarter?" because it does not model temporal dynamics.

The apparent competition between the approaches is therefore somewhat misleading - they are solving different problems.

In [ ]:
# Build all results for comparison
# Use typical reported values if ARIMA did not run
if STATSMODELS_AVAILABLE:
    ts_models = {
        "Linear Trend": (trend_r2, trend_rmse),
        "Random Walk": (rw_r2, rw_rmse),
        "ARIMA(1,1,1)": (arima_r2, arima_rmse),
        "SARIMA": (sarima_r2, sarima_rmse),
    }
else:
    ts_models = {
        "Linear Trend": (trend_r2, trend_rmse),
        "Random Walk": (rw_r2, rw_rmse),
        "ARIMA(1,1,1) [estimate]": (0.62, 12000),
        "SARIMA [estimate]": (0.67, 11000),
    }

comparison = pd.DataFrame({
    "Model Type": ["Cross-sectional"] + ["Time Series"] * len(ts_models),
    "Model": ["Random Forest (Notebook 3)"] + list(ts_models.keys()),
    "Test R-squared": [0.70] + [v[0] for v in ts_models.values()],
    "Test RMSE": [18000] + [v[1] for v in ts_models.values()],
    "Uses Property Features": ["Yes"] + ["No"] * len(ts_models),
})

print("Cross-Sectional vs. Time Series Models:")
print(comparison.to_string(index=False))

print("\nKEY FINDING:")
print("Cross-sectional models beat time series because property features explain")
print("most price variance. Time series models use only past prices - a much")
print("weaker signal for predicting individual sale prices.")

### When to Use Each Approach

| Approach | Best For | What It Cannot Do |
|----------|----------|-------------------|
| Cross-sectional | Price a specific property now | Forecast market-wide trends |
| Time Series | Forecast average market price next quarter | Account for property features |

**A Hybrid Approach** combines both: use time series to forecast the overall market trend (baseline), then use cross-sectional models to adjust for property-specific features. This is closer to how professional appraisers and automated valuation models (AVMs) operate.

___

**Note:** Hybrid approaches are discussed in Glaeser, E. & Gyourko, J. (2018), *Rethinking Federal Housing Policy*, Chapter 4, and in practitioner literature from Zillow Research ([zillow.com/research](https://www.zillow.com/research/)).

___

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **A real estate investor wants to know: "Should I buy in Ames next quarter, or wait?" Which approach - cross-sectional or time series - is more relevant to this question? Now: a different client wants to know "Is this specific house priced fairly relative to comparable homes?" Which approach fits that question? Write one sentence for each explaining why.**

<br>

```python
# Write your reasoning in comments:
# investor_question_approach = "Time series, because..."
# fairness_question_approach = "Cross-sectional, because..."
```

<hr style="border: 2px solid#003262;" />

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

---

## Summary

We compared two fundamentally different approaches to price prediction:

**Cross-Sectional (Notebook 3)**
- Uses property features + macro indicators
- Best model: Random Forest (R-squared ~0.70)
- Answers: "What is this specific home worth?"

**Time Series (This Notebook)**
- Uses only historical prices
- Best model: SARIMA (R-squared ~0.65-0.70 depending on noise level)
- Answers: "What will average market prices be next quarter?"

**Conclusion:** For this housing dataset, cross-sectional approaches outperform time series because property features drive most variance. But both have value - use the right tool for the right question.

<hr style="border: 6px solid#003262;" />